# Моделирование: прогноз суточного PM2.5

**Вопрос.** Насколько точно можно предсказать среднесуточную концентрацию PM2.5 по вчерашнему значению, погоде и календарю, и лучше ли модели простого правила «завтра как сегодня»?

**Цель:** `pm25_mean` (µg/m³) из `data/processed/daily.csv`.

**Модели (L10–L11):**
- базовые ориентиры: **наивный прогноз** («завтра как сегодня» — `pm25_lag1`) и **среднее по месяцу** обучающей выборки (`month_mean`, ориентир для моделей без лага);
- **Linear Regression** (`StandardScaler` + `LinearRegression` в `Pipeline`) и **Random Forest** (`RandomForestRegressor`) — каждая в двух вариантах: со вчерашним `pm25_lag1` и без него («только погода и календарь»).

**Почему разбиение по времени, а не `train_test_split(shuffle=True)`:** соседние сутки сильно связаны по погоде и загрязнению. Случайное перемешивание перемешало бы прошлое и будущее: в обучении оказались бы дни рядом с тестовыми, что дало бы подглядывание в будущее и заниженную ошибку.

**Почему тест трогаем один раз:** если подбирать признаки, дату разбиения или гиперпараметры по метрикам теста, тестовая часть перестаёт быть независимой проверкой. Все решения принимаются только на `train` (через скользящую проверку по времени), а на `test` модели оцениваются один раз.

**Метрики:**
- `MAE` = среднее `|факт − прогноз|` (µg/m³, основная метрика);
- `RMSE` = √(среднее `(факт − прогноз)²`) — сильнее штрафует крупные промахи;
- `R²` = `1 − Σ(факт − прогноз)² / Σ(факт − среднее факта)²` — доля разброса, которую объясняет прогноз (может быть `< 0`).

**Оговорка о погоде:** в признаках используется фактическая суточная погода дня `D`, а не метеопрогноз. Поэтому результаты показывают «оценку сверху» для настоящего прогноза на завтра.

In [1]:
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sys.path.append("..")
from src import features, hypotheses, models

DAILY_PATH = "../data/processed/daily.csv"
FIGURES_DIR = "../figures"

# Целевая переменная: среднесуточный PM2.5 (µg/m³)
TARGET = "pm25_mean"

# Полный набор признаков (погода, календарные флаги и вчерашний PM2.5)
FEATURES = [
    "temp_c",
    "wind_speed",
    "humidity",
    "precip",
    "pressure",
    "heating_season",
    "is_weekend",
    "pm25_lag1",
]

# Набор без вчерашнего PM2.5: проверяем вклад только погоды и календаря
FEATURES_NO_LAG = [
    "temp_c",
    "wind_speed",
    "humidity",
    "precip",
    "pressure",
    "heating_season",
    "is_weekend",
]

# Граница разбиения: с 01.07.2024 начинается тестовый период с полным отопительным сезоном 2024/25
TEST_START = "2024-07-01"

# Число фолдов скользящей проверки по времени на train
N_SPLITS = 5

# Фиксируем зерно случайности для леса, bootstrap и перестановочной важности
SEED = 42
N_BOOT = 5000
N_REPEATS = 30

# Единая тема графиков: палитра, удобная для дальтоников
sns.set_theme(style="whitegrid", context="talk", palette="colorblind")

## Шаг 1. Таблица для модели

Обе модели (Linear Regression и Random Forest) учатся на одном наборе признаков, чтобы их сравнение было честным: метеопараметры дня (`temp_c`, `wind_speed`, `humidity`, `precip`, `pressure`), календарные флаги (`heating_season`, `is_weekend`) и вчерашний уровень загрязнения (`pm25_lag1`). Булевы флаги переводятся в `0.0` и `1.0`.

**Почему не берём часть колонок из `daily.csv`:**
- `month` (1…12) и `dayofweek` (0…6) как числа бессмысленны для линейной модели: декабрь (`12`) и январь (`1`) одинаково грязные, но стоят на разных концах шкалы, а коэффициент «на +1 месяц» нельзя объяснить. Сезонность уже несут `heating_season` и `temp_c`, а разницу будних и выходных дней — `is_weekend`.
- `is_calm` — это жёсткий порог от `wind_speed` (`< 1 м/с`), то есть дубль скорости ветра.
- `season` (текст) потребовал бы one-hot кодирования, тогда как сезонность уже учтена.

**Почему удаляем строки с пропусками, а не заполняем:** модели должны учиться и проверяться только на реальных измеренных сутках без выдуманных значений. Все шесть прогнозов (включая модели без лага) считаются на одних и тех же строках, чтобы сравнивать их на одинаковых сутках.

**Почему `pm25_lag1` не даёт утечки данных:** по построению в `features.lag_previous_day` это среднее PM2.5 за предыдущие календарные сутки (`D−1`), известное до начала дня `D`.

In [2]:
daily = features.read_daily_csv(DAILY_PATH)
features.check_daily(daily)

n_total = len(daily)
n_no_target = int(daily[TARGET].isna().sum())
n_no_lag_only = int((daily[TARGET].notna() & daily["pm25_lag1"].isna()).sum())

table = models.make_model_table(daily, FEATURES, TARGET)
assert len(table) == 1414, f"ожидалось 1414 строк, получено {len(table)}"

print(f"Всего суток в daily.csv: {n_total}")
print(f"Без цели (pm25_mean): {n_no_target}")
print(f"С целью, но без вчерашнего значения (pm25_lag1): {n_no_lag_only}")
print(f"Всего потеряно строк: {n_total - len(table)}")
print(f"Строк в таблице модели: {len(table)}")

Всего суток в daily.csv: 2044
Без цели (pm25_mean): 510
С целью, но без вчерашнего значения (pm25_lag1): 120
Всего потеряно строк: 630
Строк в таблице модели: 1414


## Шаг 2. Разбиение по времени

Разбиваем таблицу по границе `TEST_START = "2024-07-01"`: все сутки строго до неё идут в обучение (`train`), начиная с неё — в тест (`test`).

**Почему `2024-07-01`, а не `2025-07-01`:** данные станции заканчиваются `2025-11-13` (после `2025-11-14` станция не передаёт измерения). При границе `2025-07-01` в тесте остались бы только лето, осень и начало отопительного сезона без зимы — тогда модели главной проблемы (зимнего смога) не проверялись бы. Граница `2024-07-01` оставляет в `test` полный отопительный сезон `2024/25` вместе с летом и осенью `2025` года, а в `train` — данные `2020–2024` годов.

Решение о границе разбиения принято **до** расчёта каких-либо метрик на тесте.

In [3]:
train, test = models.split_by_date(table, TEST_START)

test_share = len(test) / len(table)
train_heating = int((train["heating_season"] == 1.0).sum())
test_heating = int((test["heating_season"] == 1.0).sum())

# Считаем, сколько календарных суток в тестовом периоде не попало в таблицу модели
test_calendar = daily.loc[daily["date"] >= pd.Timestamp(TEST_START), "date"]
test_missing_days = len(test_calendar) - len(test)

print(
    f"train: n = {len(train)} ({1 - test_share:.1%}), "
    f"{train['date'].min():%Y-%m-%d} ... {train['date'].max():%Y-%m-%d}, "
    f"отопительных суток: {train_heating}"
)
print(
    f"test:  n = {len(test)} ({test_share:.1%}), "
    f"{test['date'].min():%Y-%m-%d} ... {test['date'].max():%Y-%m-%d}, "
    f"отопительных суток: {test_heating}"
)
print(
    f"Календарных суток в периоде теста: {len(test_calendar)}, "
    f"пропущено в таблице модели: {test_missing_days}"
)

train: n = 931 (65.8%), 2020-04-11 ... 2024-06-30, отопительных суток: 504
test:  n = 483 (34.2%), 2024-07-01 ... 2025-11-13, отопительных суток: 213
Календарных суток в периоде теста: 501, пропущено в таблице модели: 18


**Что видим.** Из `2 044` суток в `daily.csv` потеряно `630` строк: `510` суток без цели `pm25_mean` и ещё `120` суток с целью, но без вчерашнего значения `pm25_lag1` (вывод шага 1). В итоговой таблице модели остаётся `1 414` суток с полным набором признаков и целью. После разбиения по границе `2024-07-01` обучающая часть `train` содержит `931` сутки (`65.8%`, `2020-04-11 … 2024-06-30`, из них `504` отопительных суток), а тестовая часть `test` — `483` суток (`34.2%`, `2024-07-01 … 2025-11-13`, из них `213` отопительных суток; вывод шага 2). Из `501` календарных суток тестового периода в таблицу модели не попали `18` суток из-за пропусков цели или лага (вывод шага 2).